In [ ]:
!pip -q install "transformers==4.51.3" "sentence-transformers==3.0.1" "accelerate==1.6.0" \
        qdrant-client tqdm packaging

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 90.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.1/227.1 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.7/354.7 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 97.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


# config

In [ ]:

import os
import json
import csv
import time
import logging
import gc
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Any

import numpy as np
import pandas as pd
from tqdm import tqdm
from qdrant_client import QdrantClient
from qdrant_client.models import ScrollRequest
from google.colab import userdata


FINAL_DATASET_PATH = "dev_dataset.json"
OUTPUT_DIR = "embedding_eval_output"

TOP_K = 10
DENSE_TOP_N = 10

# Batch size dùng khi encode query.
# GTE chạy qua SentenceTransformer với phiên bản transformers đã pin ở cell cài đặt.
MODEL_BATCH_SIZES = {
    "bge_m3": 32,
    "aiteamvn": 32,
    "aiteamvn_v2": 32,
    "gte_multilingual": 32,
}

QUERY_TEXT_FIELD = "claim"

QDRANT_COLLECTIONS = {
    "bge_m3": "dvsktt_bge_m3_chunks_256",
    "aiteamvn": "dvsktt_aiteamvn_chunks_256",
    "aiteamvn_v2": "dvsktt_aiteamvn_v2_chunks_256",
    "gte_multilingual": "dvsktt_gte_multilingual_chunks_256",
}

EMBEDDING_MODELS = {
    "gte_multilingual": "Alibaba-NLP/gte-multilingual-base",
    "bge_m3": "BAAI/bge-m3",
    "aiteamvn": "AITeamVN/Vietnamese_Embedding",
    "aiteamvn_v2": "AITeamVN/Vietnamese_Embedding_v2",
}

METHOD_DISPLAY_NAMES = {
    "bge_m3": "BGE-M3",
    "aiteamvn": "AITeamVN",
    "aiteamvn_v2": "AITeamVN-v2",
    "gte_multilingual": "GTE-Multilingual",
}


logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger(__name__)



def save_json(data: Any, path: str) -> None:
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    logger.info(f"Đã lưu JSON: {path}")


def load_json(path: str) -> Any:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_jsonl(records: List[Dict], path: str) -> None:
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for rec in records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    logger.info(f"Đã lưu JSONL: {path} ({len(records)} dòng)")


def load_jsonl(path: str) -> List[Dict]:
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


def save_csv(rows: List[Dict], path: str, fieldnames: List[str]) -> None:
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    logger.info(f"Đã lưu CSV: {path}")





# kết nối qdrant
def get_qdrant_client():
    try:

        qdrant_url = userdata.get("QDRANT_URL")
        qdrant_api_key = userdata.get("QDRANT_API_KEY")
    except Exception:
        qdrant_url = os.environ.get("QDRANT_URL")
        qdrant_api_key = os.environ.get("QDRANT_API_KEY")

    if not qdrant_url or not qdrant_api_key:
        raise ValueError(
            "Thiếu QDRANT_URL hoặc QDRANT_API_KEY. "
            "Hãy thêm vào Colab Secrets hoặc os.environ."
        )


    client = QdrantClient(url=qdrant_url, api_key=qdrant_api_key, timeout=60)
    logger.info(f"Đã kết nối Qdrant: {qdrant_url}")
    return client


def scroll_corpus(client, collection_name: str) -> Dict[str, Dict]:
    """
    Scroll toàn bộ point trong collection, trả về dict chunk_id -> metadata.
    point.id là chunk_id (dùng str).
    """


    logger.info(f"Đang scroll corpus từ collection: {collection_name}")
    corpus_cache = {}
    offset = None
    batch_size = 256
    total = 0

    with tqdm(desc="Scroll Qdrant", unit=" points") as pbar:
        while True:
            try:
                results, next_offset = client.scroll(
                    collection_name=collection_name,
                    limit=batch_size,
                    offset=offset,
                    with_payload=True,
                    with_vectors=False,
                )
            except Exception as e:
                logger.error(f"Lỗi scroll Qdrant: {e}")
                raise

            if not results:
                break

            for point in results:
                chunk_id = str(point.id)
                payload = point.payload or {}
                corpus_cache[chunk_id] = {
                    "chunk_id": chunk_id,
                    "raw_text": payload.get("raw_text", ""),
                    "overlap_text": payload.get("overlap_text", ""),
                    "book_name": payload.get("book_name", ""),
                    "pages": payload.get("pages", []),
                    "footnotes": payload.get("footnotes", {}),
                    "section_title": payload.get("section_title", {}),
                    "token_count": payload.get("token_count", 0),
                }

            total += len(results)
            pbar.update(len(results))

            if next_offset is None:
                break
            offset = next_offset

    logger.info(f"Tổng số point trong corpus: {total}")
    return corpus_cache

# Kiểm tra tất cả evidence_ids có trong corpus_cache.
# Nếu thiếu thì in ra và dừng chương trình.
def validate_evidence_ids(qrels: Dict[str, List[str]], corpus_cache: Dict[str, Dict]) -> None:

    missing = set()
    for claim_id, evidence_ids in qrels.items():
        for eid in evidence_ids:
            if eid not in corpus_cache:
                missing.add(eid)

    if missing:
        logger.error(f"Có {len(missing)} evidence_id không tồn tại trong corpus:")
        for eid in sorted(missing):
            logger.error(f"  THIẾU: {eid}")
        raise SystemExit(
            f"Dừng chương trình: {len(missing)} evidence_id không tìm thấy trong Qdrant corpus."
        )

    logger.info("Tất cả evidence_ids đều tồn tại trong corpus. OK.")




# load, lọc

In [ ]:
# Load final_dataset.json, chỉ giữ label SUPPORTED và REFUTED.
#     Validate các trường bắt buộc.
def load_and_filter_dataset(path: str) -> List[Dict]:
    logger.info(f"Đang load dataset: {path}")
    raw = load_json(path)

    filtered = []
    skipped_label = 0
    skipped_invalid = 0

    for item in raw:
        label = item.get("label", "")
        if label not in ("SUPPORTED", "REFUTED"):
            skipped_label += 1
            continue

        # Validate các trường bắt buộc
        missing = [f for f in ("claim_id", "claim", "label", "evidence_ids") if not item.get(f)]
        if missing:
            logger.warning(f"Bỏ qua item thiếu trường {missing}: {item.get('claim_id', '?')}")
            skipped_invalid += 1
            continue

        filtered.append(item)

    logger.info(
        f"Dataset: tổng={len(raw)}, giữ={len(filtered)}, "
        f"bỏ NOT_ENOUGH_EVIDENCE={skipped_label}, bỏ không hợp lệ={skipped_invalid}"
    )
    return filtered


def build_qrels(dataset: List[Dict]) -> Dict[str, List[str]]:
    """qrels[claim_id] = [evidence_id, ...]"""
    qrels = {}
    for item in dataset:
        qrels[item["claim_id"]] = item["evidence_ids"]
    return qrels


# metric

In [ ]:

#  metric
def recall_at_k(gold_ids: List[str], pred_ids: List[str], k: int) -> float:
    """Recall@K = số gold_id xuất hiện trong top-k / tổng số gold_id."""
    if not gold_ids:
        return 0.0
    top_k = pred_ids[:k]
    hits = sum(1 for gid in gold_ids if gid in top_k)
    return hits / len(gold_ids)


def mrr_at_k(gold_ids: List[str], pred_ids: List[str], k: int) -> float:
    """MRR@K = 1 / rank đầu tiên của gold evidence trong top-k."""
    gold_set = set(gold_ids)
    for rank, pid in enumerate(pred_ids[:k], start=1):
        if pid in gold_set:
            return 1.0 / rank
    return 0.0


def evaluate_run(
    run: Dict[str, List[str]],
    qrels: Dict[str, List[str]],
    k_values: List[int] = (1, 3, 5, 10),
    mrr_k: int = 10,
) -> Dict[str, float]:
    """
    Tính trung bình Recall@K và MRR@K cho toàn bộ claim.

    Parameters
    ----------
    run   : {claim_id: [chunk_id, ...]} — danh sách chunk_id trả về theo thứ tự rank
    qrels : {claim_id: [evidence_id, ...]} — gold evidence
    """
    recall_sums = {k: 0.0 for k in k_values}
    mrr_sum = 0.0
    n = 0

    for claim_id, gold_ids in qrels.items():
        pred_ids = run.get(claim_id, [])
        for k in k_values:
            recall_sums[k] += recall_at_k(gold_ids, pred_ids, k)
        mrr_sum += mrr_at_k(gold_ids, pred_ids, mrr_k)
        n += 1

    if n == 0:
        return {}

    metrics = {f"R@{k}": round(recall_sums[k] / n, 4) for k in k_values}
    metrics[f"MRR@{mrr_k}"] = round(mrr_sum / n, 4)
    return metrics



# retrival

In [ ]:

# retrival

def load_embedding_model(model_name: str):

    from sentence_transformers import SentenceTransformer
    logger.info(f"Đang load embedding model: {model_name}")
    model = SentenceTransformer(model_name, trust_remote_code=True)
    logger.info(f"Đã load model: {model_name}")
    return model


def encode_queries(model, queries: List[str], batch_size: int = 32) -> np.ndarray:
    """Encode danh sách query thành vector."""
    logger.info(f"Encoding {len(queries)} queries với batch_size={batch_size}...")
    embeddings = model.encode(
        queries,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    return embeddings

# Tìm kiếm vector trên Qdrant.
# Trả về [(chunk_id, score), ...].
def dense_search_qdrant(
    client,
    collection_name: str,
    query_vector: List[float],
    top_n: int,
) -> List[Tuple[str, float]]:
    try:
        response = client.query_points(
            collection_name=collection_name,
            query=query_vector,
            limit=top_n,
            with_payload=False,
            with_vectors=False,
        )

        points = response.points
        return [(str(p.id), float(p.score)) for p in points]

    except Exception as e:
        logger.error(f"Lỗi dense search: {e}")
        return []


def run_dense_retrieval(
    client,
    dataset: List[Dict],
    model,
    collection_name: str,
    top_n: int,
    query_field: str = "claim",
    batch_size: int = 32,
) -> Dict[str, List[Tuple[str, float]]]:
    """
    Chạy dense retrieval cho toàn bộ dataset.
    Trả về {claim_id: [(chunk_id, score), ...]}
    """
    queries = [item[query_field] for item in dataset]
    claim_ids = [item["claim_id"] for item in dataset]

    embeddings = encode_queries(model, queries, batch_size=batch_size)

    results = {}
    logger.info(f"Đang dense search trên collection: {collection_name}")
    for claim_id, vec in tqdm(
        zip(claim_ids, embeddings), total=len(claim_ids), desc="Dense search"
    ):
        hits = dense_search_qdrant(client, collection_name, vec.tolist(), top_n)
        results[claim_id] = hits

    return results


def run_records_to_run_dict(run_records: List[Dict]) -> Dict[str, List[str]]:
    """Chuyển file run JSONL đã lưu thành dạng {claim_id: [chunk_id, ...]}."""
    run_dict = {}
    for rec in run_records:
        claim_id = rec["claim_id"]
        retrieved = rec.get("retrieved", [])
        retrieved = sorted(retrieved, key=lambda x: x.get("rank", 10**9))
        run_dict[claim_id] = [str(x["chunk_id"]) for x in retrieved]
    return run_dict


def free_model_memory(model=None) -> None:
    """Giải phóng RAM/GPU sau mỗi model."""
    if model is not None:
        del model
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
    except Exception:
        pass


# chọn model tốt
def select_best_model(
    metrics_table: List[Dict],
    primary_metric: str = "R@5",
    secondary_metric: str = "MRR@10",
) -> Dict:
    """Chọn best model theo primary_metric, tie-break bằng secondary_metric."""
    best = None
    for row in metrics_table:
        if best is None:
            best = row
            continue
        if row[primary_metric] > best[primary_metric]:
            best = row
        elif row[primary_metric] == best[primary_metric]:
            if row[secondary_metric] > best[secondary_metric]:
                best = row
    return best


# main

In [ ]:


def main():
    # Tạo thư mục output
    for subdir in ["retrieval_runs", "tables"]:
        Path(f"{OUTPUT_DIR}/{subdir}").mkdir(parents=True, exist_ok=True)

    #Load và lọc dataset
    dataset = load_and_filter_dataset(FINAL_DATASET_PATH)
    qrels = build_qrels(dataset)
    save_json(qrels, f"{OUTPUT_DIR}/qrels.json")
    logger.info(f"Số claim sau lọc: {len(dataset)}")

    # Kết nối Qdrant
    client = get_qdrant_client()

    # Scroll corpus từ collection bge_m3 (reference collection)
    # Dùng làm corpus_cache chung cho mọi đánh giá
    corpus_cache_path = f"{OUTPUT_DIR}/corpus_cache.json"

    if Path(corpus_cache_path).exists():
        logger.info(f"Đang load corpus_cache từ cache: {corpus_cache_path}")
        corpus_cache = load_json(corpus_cache_path)
        logger.info(f"Corpus cache: {len(corpus_cache)} chunk")
    else:
        ref_collection = QDRANT_COLLECTIONS["bge_m3"]
        corpus_cache = scroll_corpus(client, ref_collection)
        save_json(corpus_cache, corpus_cache_path)

    # Validate evidence_ids
    validate_evidence_ids(qrels, corpus_cache)

    # Đánh giá từng embedding model
    metrics_table = []

    for model_key, model_name in EMBEDDING_MODELS.items():
        display_name = METHOD_DISPLAY_NAMES.get(model_key, model_key)
        collection_name = QDRANT_COLLECTIONS[model_key]
        run_path = f"{OUTPUT_DIR}/retrieval_runs/{model_key}_top{DENSE_TOP_N}.jsonl"

        logger.info(f"\n{'='*60}")
        logger.info(f"Đang đánh giá: {display_name} ({model_name})")
        logger.info(f"Collection: {collection_name}")
        logger.info(f"{'='*60}")

        # Nếu run đã tồn tại thì load lại để không phải encode/query lại từ đầu.
        # Trường hợp notebook bị lỗi giữa chừng, các model đã chạy xong sẽ được tái sử dụng.
        if Path(run_path).exists():
            logger.info(f"Run đã tồn tại, load lại: {run_path}")
            run_records = load_jsonl(run_path)
            run_dict = run_records_to_run_dict(run_records)
        else:
            model = None
            try:
                # Load model
                model = load_embedding_model(model_name)
                batch_size = MODEL_BATCH_SIZES.get(model_key, 32)

                # Chạy dense retrieval
                retrieval_results = run_dense_retrieval(
                    client=client,
                    dataset=dataset,
                    model=model,
                    collection_name=collection_name,
                    top_n=DENSE_TOP_N,
                    query_field=QUERY_TEXT_FIELD,
                    batch_size=batch_size,
                )

                # Lưu run JSONL
                run_records = []
                run_dict = {}  # claim_id -> [chunk_id, ...]

                for item in dataset:
                    claim_id = item["claim_id"]
                    hits = retrieval_results.get(claim_id, [])
                    run_dict[claim_id] = [h[0] for h in hits]

                    run_records.append({
                        "claim_id": claim_id,
                        "claim": item[QUERY_TEXT_FIELD],
                        "label": item["label"],
                        "gold_evidence_ids": qrels[claim_id],
                        "retrieved": [
                            {"rank": rank + 1, "chunk_id": cid, "score": score}
                            for rank, (cid, score) in enumerate(hits)
                        ],
                    })

                save_jsonl(run_records, run_path)
            finally:
                free_model_memory(model)

        # Tính metric
        metrics = evaluate_run(
            run=run_dict,
            qrels=qrels,
            k_values=[1, 3, 5, 10],
            mrr_k=TOP_K,
        )

        logger.info(f"Kết quả {display_name}: {metrics}")

        metrics_table.append({
            "Method": display_name,
            "ModelName": model_name,
            "R@1": metrics.get("R@1", 0.0),
            "R@3": metrics.get("R@3", 0.0),
            "R@5": metrics.get("R@5", 0.0),
            "R@10": metrics.get("R@10", 0.0),
            "MRR@10": metrics.get("MRR@10", 0.0),
        })


    # Bước 6: Lưu bảng so sánh
    fieldnames = ["Method", "ModelName", "R@1", "R@3", "R@5", "R@10", "MRR@10"]
    table_path = f"{OUTPUT_DIR}/tables/embedding_model_comparison.csv"
    save_csv(metrics_table, table_path, fieldnames)

    # In bảng ra console
    logger.info("\n" + "=" * 70)
    logger.info("BẢNG SO SÁNH EMBEDDING MODELS")
    logger.info("=" * 70)
    df = pd.DataFrame(metrics_table)
    print(df.to_string(index=False))

    # Chọn best model
    best_row = select_best_model(metrics_table, primary_metric="R@5", secondary_metric="MRR@10")

    # Map display name ngược về model_key
    name_to_key = {v: k for k, v in METHOD_DISPLAY_NAMES.items()}
    best_key = name_to_key.get(best_row["Method"], "bge_m3")

    best_dense_config = {
        "best_dense_key": best_key,
        "best_dense_name": best_row["Method"],
        "model_name": EMBEDDING_MODELS[best_key],
        "collection_name": QDRANT_COLLECTIONS[best_key],
        "selected_by": "R@5_then_MRR@10",
    }

    save_json(best_dense_config, f"{OUTPUT_DIR}/best_dense_config.json")

    logger.info(f"\nBest embedding model: {best_row['Method']}")
    logger.info(f"  R@5={best_row['R@5']}, MRR@10={best_row['MRR@10']}")
    logger.info(f"  Config đã lưu: {OUTPUT_DIR}/best_dense_config.json")

    # Tổng kết
    logger.info("\n" + "=" * 70)
    logger.info("HOÀN THÀNH 01_eval_embedding_models.py")
    logger.info(f"Output: {OUTPUT_DIR}/")
    logger.info(f"  - qrels.json")
    logger.info(f"  - corpus_cache.json")
    logger.info(f"  - best_dense_config.json")
    logger.info(f"  - retrieval_runs/")
    logger.info(f"  - tables/embedding_model_comparison.csv")
    logger.info("=" * 70)


if __name__ == "__main__":
    main()

Scroll Qdrant: 5343 points [00:01, 5190.86 points/s]
/usr/local/lib/python3.12/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/55.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Dense search: 100%|██████████| 174/174 [00:04<00:00, 39.28it/s]


          Method                         ModelName    R@1    R@3    R@5   R@10  MRR@10
GTE-Multilingual Alibaba-NLP/gte-multilingual-base 0.4598 0.5460 0.5805 0.6552  0.5167
          BGE-M3                       BAAI/bge-m3 0.6034 0.7529 0.7816 0.8333  0.6831
        AITeamVN     AITeamVN/Vietnamese_Embedding 0.6609 0.7644 0.8161 0.8793  0.7229
     AITeamVN-v2  AITeamVN/Vietnamese_Embedding_v2 0.5690 0.6954 0.7241 0.7816  0.6423


In [ ]:
import shutil
from google.colab import files

shutil.make_archive("embedding_eval_output", 'zip', "/content/embedding_eval_output")
files.download("embedding_eval_output.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>